In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = "/content/drive/MyDrive/yolov11-object-detection"

# Count dataset files
train_imgs = len(os.listdir(f"{PROJECT_ROOT}/dataset/train/images"))
valid_imgs = len(os.listdir(f"{PROJECT_ROOT}/dataset/valid/images"))
test_imgs = len(os.listdir(f"{PROJECT_ROOT}/dataset/test/images"))
print(f"\nDataset: {train_imgs} train / {valid_imgs} valid / {test_imgs} test images")

Mounted at /content/drive
✅ Google Drive mounted successfully!
✅ Project folder found at: /content/drive/MyDrive/yolov11-object-detection

Project contents:
  📄 colab_notebook.py
  📄 compare_results.py
  📁 dataset
  📄 requirements.txt
  📁 runs
  📄 share_of_shelf.py
  📄 train_baseline.py
  📄 train_optimized.py

Dataset: 924 train / 40 valid / 35 test images


In [ ]:
import yaml

DATA_YAML = f"{PROJECT_ROOT}/dataset/data.yaml"

with open(DATA_YAML, 'r') as f:
    data_config = yaml.safe_load(f)

data_config['train'] = f"{PROJECT_ROOT}/dataset/train/images"
data_config['val'] = f"{PROJECT_ROOT}/dataset/valid/images"
data_config['test'] = f"{PROJECT_ROOT}/dataset/test/images"

# Write back
with open(DATA_YAML, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print("data.yaml updated with absolute Colab paths:")
print(f"   train: {data_config['train']}")
print(f"   val:   {data_config['val']}")
print(f"   test:  {data_config['test']}")
print(f"   nc:    {data_config['nc']}")
print(f"   names: {data_config['names'][:5]}... ({len(data_config['names'])} total)")

✅ data.yaml updated with absolute Colab paths:
   train: /content/drive/MyDrive/yolov11-object-detection/dataset/train/images
   val:   /content/drive/MyDrive/yolov11-object-detection/dataset/valid/images
   test:  /content/drive/MyDrive/yolov11-object-detection/dataset/test/images
   nc:    76
   names: ['q1', 'q10', 'q100', 'q103', 'q106']... (76 total)


In [3]:
!pip install ultralytics>=8.3.0 -q

TRAIN BASELINE MODEL (YOLOv11n) with provided Image Size (640x640), epoch =50, batch = 16

In [ ]:
from ultralytics import YOLO
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/yolov11-object-detection")
DATA_YAML = str(PROJECT_ROOT / "dataset" / "data.yaml")
OUTPUT_DIR = str(PROJECT_ROOT / "runs")

model = YOLO("yolo11n.pt")

# Train baseline with default settings

results = model.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project=OUTPUT_DIR,
    name="baseline",
    exist_ok=True,
    verbose=True,
)

print(f"Weights saved to: {OUTPUT_DIR}/baseline/weights/best.pt")

  TRAINING BASELINE — YOLOv11n (50 epochs)
Ultralytics 8.4.18 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/yolov11-object-detection/dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=baseline, nbs=64, nms=False, opset=None

VALIDATE BASELINE & RECORD METRICS

In [ ]:
from ultralytics import YOLO
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/yolov11-object-detection")
DATA_YAML = str(PROJECT_ROOT / "dataset" / "data.yaml")
OUTPUT_DIR = PROJECT_ROOT / "runs"

# Load the trained baseline model
baseline_weights = OUTPUT_DIR / "baseline" / "weights" / "best.pt"
model = YOLO(str(baseline_weights))

# Validate on TEST set
metrics = model.val(
    data=DATA_YAML,
    split="test",
    device=0,
    project=str(OUTPUT_DIR),
    name="baseline_val",
    exist_ok=True,
)

precision = metrics.box.mp
recall = metrics.box.mr
map50 = metrics.box.map50
map50_95 = metrics.box.map

print("\n" + "=" * 50)
print("  BASELINE RESULTS")
print("=" * 50)
print(f"  Precision:  {precision*100:.2f}%")
print(f"  Recall:     {recall*100:.2f}%")
print(f"  mAP@50:     {map50*100:.2f}%")
print(f"  mAP@50-95:  {map50_95*100:.2f}%")
print("=" * 50)

# Save metrics
metrics_file = OUTPUT_DIR / "baseline" / "baseline_metrics.txt"
with open(metrics_file, "w") as f:
    f.write(f"Precision: {precision:.4f}\n")
    f.write(f"Recall: {recall:.4f}\n")
    f.write(f"mAP@50: {map50:.4f}\n")
    f.write(f"mAP@50-95: {map50_95:.4f}\n")

print(f"\nMetrics saved to: {metrics_file}")

STRATEGY A — LOWER CONFIDENCE THRESHOLD (instant, no train)

In [ ]:
from ultralytics import YOLO
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/yolov11-object-detection")
DATA_YAML = str(PROJECT_ROOT / "dataset" / "data.yaml")
OUTPUT_DIR = PROJECT_ROOT / "runs"

# Load the trained baseline model
baseline_weights = OUTPUT_DIR / "baseline" / "weights" / "best.pt"
model = YOLO(str(baseline_weights))

print("=" * 60)
print("  STRATEGY A — Lower Confidence Threshold (conf=0.15)")
print("=" * 60)

# Validate with LOWER confidence threshold
metrics = model.val(
    data=DATA_YAML,
    split="test",
    conf=0.15,       # Lower threshold → more detections → higher recall
    device=0,
    project=str(OUTPUT_DIR),
    name="strategy_a_conf015",
    exist_ok=True,
)

precision_a = metrics.box.mp
recall_a = metrics.box.mr
map50_a = metrics.box.map50
map50_95_a = metrics.box.map

print(f"\n  Precision:  {precision_a*100:.2f}%")
print(f"  Recall:     {recall_a*100:.2f}%")
print(f"  mAP@50:     {map50_a*100:.2f}%")
print(f"  mAP@50-95:  {map50_95_a*100:.2f}%")